# FemScan AI — Cervical Classifier Training

**Before you start:** Runtime > Change runtime type > **T4 GPU**

This notebook has two parts:
- **Part A** — Single-task 6-class cytology classifier (`train_cervical.py`)
- **Part B** — Multi-modal multi-task model: cytology + HPV/clinical tabular data (`train_multimodal.py`)

### One-time Drive setup (do this before running)
Zip your `data/sipakmed/` folder and upload it to Google Drive:
```powershell
# On your Windows machine:
cd "C:\Users\ADMIN\OneDrive\Desktop\OSBORN\HACKATHON POSTINGS\femscan-ai"
Compress-Archive -Path data\sipakmed -DestinationPath sipakmed_images.zip
```
Upload `sipakmed_images.zip` to the root of your Google Drive.

## Step 1 — GPU check

In [ ]:
import torch
print('GPU available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('Device:', torch.cuda.get_device_name(0))
    print('VRAM (GB):', round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1))

## Step 2 — Clone repo + install dependencies

In [ ]:
import os
REPO_URL = 'https://github.com/Donitero/CerviScan.git'
BRANCH   = 'claude/bold-heyrovsky'
REPO_DIR = '/content/femscan-ai'

if not os.path.exists(REPO_DIR):
    !git clone --branch {BRANCH} {REPO_URL} {REPO_DIR}
else:
    !cd {REPO_DIR} && git pull

%cd {REPO_DIR}
print('Repo ready at', REPO_DIR)

In [ ]:
!pip install -q \
    timm>=0.9.12 \
    albumentations>=1.3.1 \
    opencv-python-headless>=4.8.0 \
    scikit-learn>=1.3.0 \
    pandas>=2.1.0 \
    pytorch-grad-cam>=1.5.0 \
    seaborn
print('Dependencies installed.')

## Step 3 — Mount Drive and unzip images

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
print('Drive mounted.')

In [ ]:
import zipfile
from pathlib import Path

ZIP_PATH = '/content/drive/MyDrive/sipakmed_images.zip'
DEST     = f'{REPO_DIR}/data/sipakmed'

if not Path(ZIP_PATH).exists():
    raise FileNotFoundError(
        f'Could not find {ZIP_PATH}\n'
        'Upload sipakmed_images.zip to the root of your Google Drive first.'
    )

print(f'Unzipping {ZIP_PATH} ...')
with zipfile.ZipFile(ZIP_PATH, 'r') as zf:
    zf.extractall(f'{REPO_DIR}/data')

for c in sorted(Path(DEST).iterdir()):
    if c.is_dir():
        print(f'  {c.name}: {len(list(c.glob("*.png")))} images')
print('Done.')

## Step 4 — Rebuild split manifest
The repo manifest uses Windows paths. This cell rebuilds it for `/content/femscan-ai/...`.

In [ ]:
import csv
from pathlib import Path
from sklearn.model_selection import train_test_split

PROJ    = Path(REPO_DIR)
SRC     = PROJ / 'data' / 'sipakmed'
OUT_CSV = PROJ / 'data' / 'sipakmed_split.csv'
SEED    = 42

rows = []
for cls_dir in sorted(SRC.iterdir()):
    if not cls_dir.is_dir(): continue
    imgs = sorted(cls_dir.glob('*.png'))
    n = len(imgs)
    if n < 3:
        for p in imgs:
            rows.append({'filepath': str(p.relative_to(PROJ)), 'class': cls_dir.name, 'split': 'train'})
        continue
    tr, tmp = train_test_split(imgs, test_size=0.30, random_state=SEED)
    va, te  = train_test_split(tmp,  test_size=0.50, random_state=SEED) if len(tmp) >= 2 else (tmp, [])
    for p in tr: rows.append({'filepath': str(p.relative_to(PROJ)), 'class': cls_dir.name, 'split': 'train'})
    for p in va: rows.append({'filepath': str(p.relative_to(PROJ)), 'class': cls_dir.name, 'split': 'val'})
    for p in te: rows.append({'filepath': str(p.relative_to(PROJ)), 'class': cls_dir.name, 'split': 'test'})

with open(OUT_CSV, 'w', newline='') as f:
    w = csv.DictWriter(f, fieldnames=['filepath', 'class', 'split'])
    w.writeheader(); w.writerows(rows)

from collections import Counter
sp = Counter(r['split'] for r in rows)
print(f'Manifest: {len(rows)} rows  train={sp["train"]}  val={sp["val"]}  test={sp["test"]}')

---
# Part A — Single-task cytology classifier
6-class Bethesda classification (Negative / ASC-US / LSIL / ASC-H / HSIL / ca)
with class-weighted loss + balanced sampler.

## Step 5A — Visualise augmented samples

In [ ]:
import sys, torch, numpy as np
import matplotlib.pyplot as plt
from torch.utils.data import DataLoader
sys.path.insert(0, REPO_DIR)

from scripts.train_cervical import CervicalDataset, build_train_transform
from models.cervical_classifier import CLASS_NAMES

train_ds = CervicalDataset(str(OUT_CSV), 'train', build_train_transform(224))
imgs, labels = next(iter(DataLoader(train_ds, batch_size=8, shuffle=True)))

mean = torch.tensor([0.485, 0.456, 0.406]).view(3,1,1)
std  = torch.tensor([0.229, 0.224, 0.225]).view(3,1,1)
disp = (imgs * std + mean).clamp(0, 1)

fig, axes = plt.subplots(2, 4, figsize=(14, 7))
for i, ax in enumerate(axes.flat):
    ax.imshow(disp[i].permute(1,2,0).numpy())
    ax.set_title(CLASS_NAMES[labels[i]], fontsize=9)
    ax.axis('off')
plt.suptitle('Augmented training samples'); plt.tight_layout(); plt.show()

## Step 6A — Train single-task classifier

In [ ]:
EPOCHS        = 25
BATCH_SIZE    = 16
FREEZE_EPOCHS = 5
LR            = 3e-4

!python {REPO_DIR}/scripts/train_cervical.py \
    --manifest      {REPO_DIR}/data/sipakmed_split.csv \
    --out-dir       {REPO_DIR}/trained_models \
    --epochs        {EPOCHS} \
    --batch-size    {BATCH_SIZE} \
    --freeze-epochs {FREEZE_EPOCHS} \
    --lr            {LR}

## Step 7A — Evaluate single-task model on test set

In [ ]:
import torch, sys, numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from torch.utils.data import DataLoader
from sklearn.metrics import classification_report, confusion_matrix
sys.path.insert(0, REPO_DIR)

from scripts.train_cervical import CervicalDataset, build_val_transform
from models.cervical_classifier import CervicalClassifier, CLASS_NAMES

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model  = CervicalClassifier(
    pretrained=False,
    checkpoint_path=f'{REPO_DIR}/trained_models/cervical_best.pt'
).to(device)
model.eval()

test_ds     = CervicalDataset(str(OUT_CSV), 'test', build_val_transform(224))
test_loader = DataLoader(test_ds, batch_size=16, shuffle=False)

all_preds, all_labels = [], []
with torch.no_grad():
    for imgs, labels in test_loader:
        preds = model(imgs.to(device)).argmax(1).cpu()
        all_preds.extend(preds.tolist())
        all_labels.extend(labels.tolist())

print(classification_report(all_labels, all_preds, target_names=CLASS_NAMES, zero_division=0))

cm = confusion_matrix(all_labels, all_preds)
fig, ax = plt.subplots(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES, ax=ax)
ax.set_title('Single-task — Confusion Matrix (Test Set)')
ax.set_xlabel('Predicted'); ax.set_ylabel('True')
plt.tight_layout(); plt.show()

---
# Part B — Multi-Modal Multi-Task Model

Architecture:
```
[Cytology image]  -> EfficientNetV2-S  -> 1280-d
                                              |
[Clinical CSV]    -> MLP + HPV embed   ->  64-d   -> Fusion -> progression_risk  (sigmoid)
                                              |              -> cancer_probability (sigmoid)
[Colposcopy]*     -> EfficientNet-B0   -> 512-d
```
*optional — skip if no colposcopy images

Labels derived from Bethesda class:
- `progression_label = 1` if class is ASC-H / HSIL / ca
- `cancer_label = 1` if class is ca

## Step 5B — Generate synthetic clinical data

In [ ]:
import sys
sys.path.insert(0, REPO_DIR)
from pathlib import Path
from scripts.generate_clinical_data import generate

clinical_path = Path(REPO_DIR) / 'data' / 'clinical' / 'clinical_data.csv'
clinical_path.parent.mkdir(parents=True, exist_ok=True)

df = generate(Path(REPO_DIR) / 'data' / 'sipakmed_split.csv')
df.to_csv(clinical_path, index=False)

print(f'Clinical data: {len(df)} rows -> {clinical_path}')
print(f'Progression positives : {df["progression_label"].sum()} / {len(df)}')
print(f'Cancer positives      : {df["cancer_label"].sum()} / {len(df)}')
print(df[['bethesda_class','HPV_type','age','progression_label','cancer_label']].groupby('bethesda_class').mean().round(2))

## Step 6B — Verify multi-modal dataset loads

In [ ]:
import sys, torch
sys.path.insert(0, REPO_DIR)
from scripts.train_multimodal import MultiModalDataset, val_tf, collate_fn
from torch.utils.data import DataLoader

manifest_path = f'{REPO_DIR}/data/sipakmed_split.csv'
clinical_path = f'{REPO_DIR}/data/clinical/clinical_data.csv'

ds = MultiModalDataset(manifest_path, clinical_path, 'train', val_tf(224))
print(f'Train samples: {len(ds)}')
print(f'Positive-class weights: {ds.pos_weights()}')

# Load one batch
loader = DataLoader(ds, batch_size=4, shuffle=True, collate_fn=collate_fn)
cyto, tabular, colpo, labels = next(iter(loader))
print('Cytology batch  :', cyto.shape)
print('Tabular keys    :', list(tabular.keys()))
print('Progression labels:', labels['progression'].tolist())
print('Cancer labels     :', labels['cancer'].tolist())

## Step 7B — Train multi-modal model

In [ ]:
# Typical T4 runtime: ~12-18 minutes for 30 epochs on 232 images
EPOCHS        = 30
BATCH_SIZE    = 16
FREEZE_EPOCHS = 5
CANCER_WEIGHT = 2.0   # up-weight cancer task (rarer label)
PATIENCE      = 7     # early stopping

!python {REPO_DIR}/scripts/train_multimodal.py \
    --manifest      {REPO_DIR}/data/sipakmed_split.csv \
    --clinical-csv  {REPO_DIR}/data/clinical/clinical_data.csv \
    --out-dir       {REPO_DIR}/trained_models \
    --epochs        {EPOCHS} \
    --batch-size    {BATCH_SIZE} \
    --freeze-epochs {FREEZE_EPOCHS} \
    --cancer-weight {CANCER_WEIGHT} \
    --patience      {PATIENCE}

## Step 8B — View training curves

In [ ]:
from PIL import Image as PILImage
import matplotlib.pyplot as plt

curves_path = f'{REPO_DIR}/trained_models/training_curves.png'
img = PILImage.open(curves_path)
plt.figure(figsize=(16, 4))
plt.imshow(img); plt.axis('off')
plt.title('Multi-modal training curves'); plt.show()

## Step 9B — Test set evaluation + confusion matrices

In [ ]:
import torch, sys, numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from torch.utils.data import DataLoader
from sklearn.metrics import classification_report, roc_auc_score
sys.path.insert(0, REPO_DIR)

from scripts.train_multimodal import MultiModalDataset, val_tf, collate_fn, compute_metrics
from models.multimodal_classifier import CervicalMultiModal

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model  = CervicalMultiModal.load(
    f'{REPO_DIR}/trained_models/multimodal_best.pt', device=str(device)
).to(device)
model.eval()

test_ds     = MultiModalDataset(
    f'{REPO_DIR}/data/sipakmed_split.csv',
    f'{REPO_DIR}/data/clinical/clinical_data.csv',
    'test', val_tf(224)
)
test_loader = DataLoader(test_ds, batch_size=16, shuffle=False, collate_fn=collate_fn)

prog_preds, prog_labels, cancer_preds, cancer_labels = [], [], [], []
with torch.no_grad():
    for cyto, tabular, colpo, labels in test_loader:
        cyto    = cyto.to(device)
        tabular = {k: v.to(device) for k, v in tabular.items()}
        out = model(cyto, tabular)
        prog_preds.extend(torch.sigmoid(out['progression_risk']).cpu().numpy())
        cancer_preds.extend(torch.sigmoid(out['cancer_probability']).cpu().numpy())
        prog_labels.extend(labels['progression'].numpy())
        cancer_labels.extend(labels['cancer'].numpy())

prog_preds   = np.array(prog_preds)
cancer_preds = np.array(cancer_preds)
prog_labels   = np.array(prog_labels)
cancer_labels = np.array(cancer_labels)

fig, axes = plt.subplots(1, 2, figsize=(10, 4))
for ax, preds, lbls, title in [
    (axes[0], prog_preds,   prog_labels,   'Progression Risk'),
    (axes[1], cancer_preds, cancer_labels, 'Cancer Probability'),
]:
    from sklearn.metrics import confusion_matrix
    cm = confusion_matrix(lbls, (preds >= 0.5).astype(int))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                xticklabels=['Neg','Pos'], yticklabels=['Neg','Pos'], ax=ax)
    m = compute_metrics(preds, lbls)
    ax.set_title(f'{title}\nF1={m["f1"]:.3f}  AUC={m["auc"]:.3f}')
    ax.set_xlabel('Predicted'); ax.set_ylabel('True')
    print(f'{title}: Acc={m["acc"]:.3f}  P={m["precision"]:.3f}  R={m["recall"]:.3f}  F1={m["f1"]:.3f}  AUC={m["auc"]:.3f}')

plt.tight_layout(); plt.show()

## Step 10 — Save all checkpoints to Drive

In [ ]:
import shutil, time
from pathlib import Path

tag      = time.strftime('%Y%m%d_%H%M')
dest_dir = Path(f'/content/drive/MyDrive/femscan_checkpoints/{tag}')
dest_dir.mkdir(parents=True, exist_ok=True)

for fname in ['cervical_best.pt', 'multimodal_best.pt',
              'training_curves.png', 'confusion_progression.png', 'confusion_cancer.png']:
    src = Path(f'{REPO_DIR}/trained_models/{fname}')
    if src.exists():
        shutil.copy2(src, dest_dir / fname)
        print(f'  Saved: {dest_dir / fname}')
    else:
        print(f'  Not found (skipping): {fname}')

print(f'\nAll checkpoints saved to Drive: {dest_dir}')